# AI in Industry — the lab

`v10` &nbsp;·&nbsp; Vignan's Foundation for Science, Technology & Research

**Nothing installs on your computer.** Everything runs on Google's servers; your laptop only needs a browser.

You will build a question-answering system over **your own B.Tech academic regulations**, one piece at a time, then break it on purpose to find out how it fails.

---

## Do these three things first

### 1. Make your own copy

**File → Save a copy in Drive**, then **close the tab you were in before.**

The two tabs look almost identical, but only your copy saves what you type. Check the title at the top left reads **`Copy of ...`** before going on.

### 2. Get a free API key

**[console.groq.com](https://console.groq.com/keys)** → sign in with Google → **API Keys** → **Create API Key** → name it `lab` → **Submit**.

Copy it. It starts `gsk_` and is **shown only once** — if you lose it, just make another. It is free.

### 3. Store the key in Colab

Do **not** paste your key into a code cell.

1. On the **far left edge**, click the **🔑 key icon**
2. **+ Add new secret**
3. **Name:** `LLM_API_KEY` — all capitals, two underscores
4. **Value:** your `gsk_...` key
5. **Notebook access:** turn the toggle **ON**

**Two things go wrong here, every time:**

- **The toggle starts OFF** and shows a grey ✕. Leave it off and nothing can read your key, even though the secret looks saved.
- **The name box hides the end of what you typed.** It can show `LLM_API` whether or not you finished. Click in, press `End`, and check it says `LLM_API_KEY`.

---

## How to use this notebook

**Work downwards, one cell at a time.** Run a cell, read the note under it, then run the next.

Cells marked `# ===== EDIT ME =====` hold **only the question or setting**. Change those, re-run the cell below them, and watch the answer move. That is the whole point — reading code teaches you very little, changing it teaches you a lot.

Use the **table of contents** (☰ top left) to jump between steps.

**If something breaks, carry on.** Skip to the next step. Nothing later depends on an earlier step having worked.

---

# Setup

Run this **once**. It takes about a minute — most of it downloading a small language model. It will look frozen. It is not.

In [ ]:
#@title Run me first - setup (about 60 seconds) { display-mode: "form" }
# Runs ONCE for the whole lab. Downloads the regulations, installs what is
# needed, and reads your API key from the Colab secrets panel.
import os, sys, pathlib, subprocess

NOTEBOOK_VERSION = 10      # bumped whenever this notebook changes

BASE = "/content" if pathlib.Path("/content").exists() else "."
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)
os.chdir(BASE)

if pathlib.Path("rough").exists():
    subprocess.run("rm -rf rough", shell=True)
print("Downloading the lab files ...")
subprocess.run("git clone --depth 1 --quiet "
               "https://github.com/coolMukul/rough.git rough", shell=True)

os.chdir(f"{BASE}/rough/ai-lab")
sys.path.insert(0, os.getcwd())

print("Installing packages - this is the slow bit, please wait ...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt",
               shell=True)

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = (userdata.get("LLM_API_KEY") or "").strip()
except Exception:
    pass

key = os.environ.get("LLM_API_KEY", "")
print()
if not key:
    print("  NO API KEY FOUND.")
    print()
    print("  1. Click the key icon on the far left edge")
    print("  2. Click '+ Add new secret'")
    print("  3. Name it exactly:  LLM_API_KEY")
    print("     The name box is narrow and hides the end of long text -")
    print("     click into it and press End to check the whole name is there.")
    print("  4. Paste your key from  https://console.groq.com/keys")
    print("  5. Turn ON the 'Notebook access' toggle. It starts OFF and")
    print("     shows a grey X. This is the most common mistake.")
    print("  6. Run this cell again.")
else:
    print(f"  Ready. Key ending ...{key[-4:]}")
    print(f"  Working in {os.getcwd()}")
    try:
        latest = int(pathlib.Path("lab-version.txt").read_text().strip())
        if NOTEBOOK_VERSION < latest:
            print()
            print(f"  NOTE: this notebook is v{NOTEBOOK_VERSION}, latest is v{latest}.")
            print("  Everything still works. For the newest version, take a")
            print("  fresh copy from the link the lecturer gave you.")
    except Exception:
        pass
    print()
    print("  Now work down the notebook one cell at a time.")

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#f9a825">&#9888; If this says NO API KEY FOUND, follow the six lines it prints and run it again. Everything below needs it.</b></div>

---

# Step 1 — The model has no memory

You will call a language model directly and discover that it remembers nothing at all between calls — not even the message you sent five seconds ago.

**What you should end up understanding:** There is no memory on the server. Everything that feels like a conversation is a Python list your own code re-sends every time.

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#f9a825">&#9888; Uses about 3 API calls. Re-running cells is fine, it just uses a little more quota.</b></div>

Everything that feels like memory is **a list your code re-sends**.

Work down the notebook one cell at a time, reading the notes between them.

## First, the whole API

One function. Everything in this lab is built on it.

In [ ]:
from labcore import chat

# chat(messages) -> the reply text. That is the entire interface.

### Your question

This cell holds **only the question**. Change it to anything you like.

In [ ]:
# ===== EDIT ME, then run the cell below =====
my_question = "In one sentence, what is a university?"

### Send it

In [ ]:
print(chat([{"role": "user", "content": my_question}]))

Go back, change `my_question`, run both cells again. That is the loop you will use all session.

---

## Now tell it something to remember

In [ ]:
# ===== EDIT ME, then run the cell below =====
first_turn = "My roll number is 24BCE0142. Remember it."

In [ ]:
print(chat([{"role": "user", "content": first_turn}]))

It probably said something like *"I've noted that for the rest of our conversation."*

**That is not true.** Watch.

In [ ]:
# ===== EDIT ME, then run the cell below =====
second_turn = "What is my roll number?"

In [ ]:
# A brand new call. Nothing connects it to the one above.
print(chat([{"role": "user", "content": second_turn}]))

It has no idea.

No session, no memory, no profile. Every call starts from nothing.

---

## So how do chatbots remember?

**Your code re-sends the whole conversation.** Here it is by hand — the list is the only thing that changed.

In [ ]:
# ===== EDIT ME, then run the cell below =====
history = [
    {"role": "user",      "content": "My roll number is 24BCE0142. Remember it."},
    {"role": "assistant", "content": "Noted, your roll number is 24BCE0142."},
    {"role": "user",      "content": "What is my roll number?"},
]

In [ ]:
print(chat(history))

Now it knows — because the answer was inside the question.

**Memory is something you implement.**

---

## Now change it yourself

Go back to the `history` cell and try each of these, re-running the cell below it each time:

- **Delete the middle message** (the `assistant` one). Does it still work?
- **Change the roll number in the first message only.** Which one does it report?
- **Add two more turns**, then ask about something from the first one.

Then run this to see what a conversation actually costs you:

In [ ]:
total = sum(len(m["content"]) for m in history)
print(f"{len(history)} messages, {total} characters re-sent on every single turn.")
print("This is why long conversations get expensive.")

---

# Step 2 — Without your data, it invents

You will ask the model about your own college's attendance rules — which it has never seen — and watch it answer anyway.

**What you should end up understanding:** A wrong answer arrives in exactly the same confident tone as a right one. You cannot tell them apart by reading.

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#f9a825">&#9888; Uses about 1 API call. Re-running cells is fine, it just uses a little more quota.</b></div>

The model has never seen your college's regulations. Ask anyway, and watch what happens.

In [ ]:
from labcore import chat

### The question

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = ("At VFSTR, what is the minimum attendance required in a course, "
            "and what happens if I fall below it?")

### Ask it, with no context at all

In [ ]:
print(chat([{"role": "user", "content": question}]))

## Now go and check it

Open your real regulations: **[R22.1 PDF](https://vignan.ac.in/2023pdf/R22.1-B.Tech%20Regulations.pdf)** — section 4 is Attendance.

Some of that answer is right. Some is invented. **They are written in exactly the same tone**, and nothing in the wording tells you which is which.

That is the problem the rest of this lab exists to solve.

---

## Now change it yourself

Ask about something you can verify, then check it in the PDF. Change the question cell and re-run the cell below it.

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = "What CGPA do I need for a First Class with Distinction at VFSTR?"

# Others worth trying:
#   "What is the condonation limit for attendance shortage at VFSTR?"
#   "How many credits do I need for a B.Tech at VFSTR?"
#   "How long do I have to finish the degree at VFSTR?"

In [ ]:
print(chat([{"role": "user", "content": question}]))

**Run that same pair two or three times without changing anything.**

You may get a different answer each time. A system that gives a different answer to the same question is not one you can build on.

---

# Step 3 — Load the regulations

You will load the 227 clauses that were extracted from your B.Tech regulations PDF, and look at what they actually contain.

**What you should end up understanding:** How a document gets turned into something searchable, and why each chunk carries its section name.

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#2e7d32">&#10003; This step is free. No API calls at all - run and re-run everything as much as you like.</b></div>

No model in this step, and **no API calls at all** — run everything as often as you like.

Your 52-page regulations PDF has already been split into 227 numbered clauses.

In [ ]:
from labcore import corpus

chunks, texts, manifest = corpus()

print(f"{len(chunks)} chunks from {manifest['source_pdf']}")

### Look at one

In [ ]:
# ===== EDIT ME, then run the cell below =====
which = 0            # try 0, 50, 120, 226

In [ ]:
print(f"page {chunks[which]['page']}   section: {chunks[which]['section']}")
print()
print(texts[which][:400])

Notice the `[Section]` prefix on the text. **Every chunk starts with its section name.**

That is not decoration. Without it, searching for "attendance" found nothing useful — retrieval failed completely in testing until it was added.

---

## Find the clauses that matter

In [ ]:
# ===== EDIT ME, then run the cell below =====
section_wanted = "attend"     # try: assess, award, grade, curriculum

In [ ]:
hits = [c for c in chunks if section_wanted.lower() in c["section"].lower()]

print(f"{len(hits)} chunks in sections matching '{section_wanted}'\n")
for c in hits[:6]:
    print(f"p{c['page']}  [{c['section']}]")
    print(f"      {c['text'][:150]}...")
    print()

---

## Now change it yourself

Change `section_wanted` and re-run. Find the clause about **condonation** — you will need it in step 6.

How many chunks does your regulations document devote to assessment, compared to attendance? That tells you something about what the university cares about.

---

# Step 4 — Find the right clause

You will build a search index over those clauses and retrieve the ones matching a question. No model is involved at all.

**What you should end up understanding:** Retrieval is ordinary search — the DBMS material you already know, with a different relevance score.

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#2e7d32">&#10003; This step is free. No API calls at all - run and re-run everything as much as you like.</b></div>

Still **no API calls**. This step is pure search.

You have done this before: documents, an index, a query, ranked results. It is the DBMS material with a different relevance score.

In [ ]:
from labcore import corpus, tfidf

chunks, texts, _ = corpus()
retrieve = tfidf(texts)          # builds the index over all 227 clauses

print("index built")

### Your search query

Only the query lives here. Change it as often as you like — this costs nothing.

In [ ]:
# ===== EDIT ME, then run the cell below =====
query = "What is the minimum attendance required?"
k = 3                  # how many chunks to bring back

### Search

In [ ]:
for text, score in retrieve(query, k=k):
    print(f"{score:.3f}   {text[:95]}...")
    print()

The top hit should be the actual rule. **No model was involved** — this is word overlap, scored and sorted.

---

## Now change it yourself

Run the pair above with each of these and watch the score:

| Query | What you should see |
|---|---|
| `condonation` | Nails it — the exact word is in the clause |
| `attendance shortage` | Works well |
| `exemption for missing class` | **Score near zero.** Nothing matched |
| `how many credits for degree` | Works |

That third one matters. **"exemption" and "condoned" mean the same thing and share no letters**, so word-matching cannot connect them. Remember it — step 6 comes back to it.

---

# Step 5 — This is RAG

You will join step 4's search to step 1's model: find the right clause, paste it into the prompt, and ask the question.

**What you should end up understanding:** That is all RAG is. Retrieve, stuff into the prompt, instruct the model to stay inside it.

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#f9a825">&#9888; Uses about 1 API call. Re-running cells is fine, it just uses a little more quota.</b></div>

Two things you already have:

- **Step 4** finds the right clause
- **Step 1** sends text to a model

Put the clause into the prompt. That is all RAG is.

In [ ]:
from labcore import corpus, tfidf, chat

chunks, texts, _ = corpus()
retrieve = tfidf(texts)

### Your question

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = ("At VFSTR, what is the minimum attendance required in a course, "
            "and what happens if I fall below it?")
k = 4                  # how many clauses to put in the prompt

### Step one: retrieve, and look at what we found

In [ ]:
hits = retrieve(question, k=k)
context = "\n\n".join(f"[{i+1}] {t}" for i, (t, _) in enumerate(hits))

print("THIS IS WHAT WE ARE ABOUT TO PASTE INTO THE PROMPT:")
print()
print(context[:900], "...")

### Step two: the instruction

This is the system prompt. It is doing real work — it is what stops the model wandering off into invention.

In [ ]:
# ===== EDIT ME, then run the cell below =====
system = ("You answer questions about VFSTR academic regulations using ONLY "
          "the context provided. If the context does not contain the answer, "
          'say exactly: "I could not find that in the regulations." '
          "Never use outside knowledge. Quote the rule you relied on.")

### Step three: send both

In [ ]:
answer = chat([
    {"role": "system", "content": system},
    {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"},
])

print(answer)

**Compare that with step 2.** Same model, same question. The only difference is the paragraph we pasted in first.

---

## Now change it yourself

Two experiments, both worth doing:

**1. Break the system prompt.** Go up to the `system` cell, delete the sentence *"If the context does not contain the answer, say exactly..."*, then ask something that is **not** in the regulations (try *"What is the hostel curfew?"*). Does it start inventing again?

**2. Change `k`.** Set it to `1` and re-run everything below. Then `8`. Watch how much text goes into the prompt, and whether the answer changes.

That second one is the whole of step 6.

---

# Step 6 — Break it

You will break the system you just built, in three different ways, and work out which part was actually at fault.

**What you should end up understanding:** Most of the time "the AI is wrong", the retrieval was wrong. This is the most useful debugging instinct in the session.

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#f9a825">&#9888; Uses about 3 API calls. Re-running cells is fine, it just uses a little more quota.</b></div>

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; This is the most important notebook in the lab. Do not skim it — read both answers in the k=1 versus k=6 cells side by side before moving on.</b></div>

You have a working RAG system. Now find out how it fails — **that is the actual job.**

Three failures, in order.

In [ ]:
from labcore import corpus, tfidf, grounded

chunks, texts, _ = corpus()
retrieve = tfidf(texts)
ask = grounded(retrieve)     # the retrieve-then-answer from step 5, in one call

print("ready")

## (a) Ask something that genuinely is not in there

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = "What are the hostel mess timings?"

In [ ]:
print(ask(question, k=4))

It refuses.

**Refusing correctly is a PASSING score**, not a failure. A system that invents an answer about your attendance is worse than useless.

---

## (b) Starve it

Now the important one. Same model, same question. **The only thing that changes is `k`.**

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = ("I have 68% attendance in one course because I was away at "
            "placement drives. What happens to me?")

### With k=1 — one clause only

In [ ]:
print(ask(question, k=1))

### With k=6 — six clauses

In [ ]:
print(ask(question, k=6))

**Read both answers side by side before going on.**

- `k=1` says you are below 75% and in trouble.
- `k=6` says 68% is *within the 10% condonable range*, and placement activity is a listed ground.

One of those would make a student panic for no reason.

**The model was not wrong. It was starved.** At `k=1` it never saw the condonation clause, so it answered correctly from half the picture — confidently, with no hint anything was missing.

> Most of the time "the AI is wrong", **the retrieval was wrong.**

That is the most useful debugging instinct in this whole session.

---

## (c) The right question in the wrong words

In [ ]:
# ===== EDIT ME, then run the cell below =====
rulebook_words = "shortage of attendance condoned"
student_words  = "exemption for missing too many classes"

In [ ]:
print("rulebook wording ->", retrieve(rulebook_words, k=1)[0][0][:85])
print()
print("student wording  ->", retrieve(student_words,  k=1)[0][0][:85])

The second retrieves something irrelevant.

**"exemption" and "condoned" mean the same thing and share no letters.** The rulebook and the student describe the same rule in different words, and word-matching cannot bridge it.

---

## Now change it yourself

**Find the value of k where the answer flips.** Change `k` in the cell below, run it, and narrow it down.

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = ("I have 68% attendance in one course because I was away at "
            "placement drives. What happens to me?")
k = 2                 # try 1, then 2, then 3... where does it change?

In [ ]:
print(ask(question, k=k))

Then write your own question about a rule that **has an exception attached to it** — those are the ones that need two clauses, and they are where this failure lives.

Your regulations are full of them. Step 3 will help you find one.

---

# Step 7 — Match on meaning, not words

You will swap the word-matching retriever for one that matches on meaning, and compare what each one finds.

**What you should end up understanding:** What embeddings are for — and that swapping in a fancier technique does not automatically make a system better.

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#f9a825">&#9888; Uses about 1 API call. Re-running cells is fine, it just uses a little more quota.</b></div>

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; Honest warning: on this corpus embeddings do NOT clearly beat the simpler method. That is a real result, not a broken notebook. It is exactly why step 11 exists.</b></div>

Step 6(c) showed word-matching failing on *"exemption"* versus *"condoned"*.

**Embeddings** turn text into a list of numbers, positioned so that similar meanings land near each other. A match no longer needs shared words.

The vectors for all 227 clauses ship with the lab, already computed.

In [ ]:
from labcore import corpus, tfidf, embed, grounded

chunks, texts, _ = corpus()
by_word    = tfidf(texts)      # step 4's retriever
by_meaning = embed(texts)      # the new one

print("both retrievers ready")

### The paraphrased question

The one that broke word-matching.

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = "Can I get an exemption if I miss too many classes?"

### Compare what each retriever finds

In [ ]:
print("BY WORD:")
for t, s in by_word(question, k=2):
    print(f"   {s:.3f}  {t[:80]}...")

print()
print("BY MEANING:")
for t, s in by_meaning(question, k=2):
    print(f"   {s:.3f}  {t[:80]}...")

# The two score scales are different. What matters is WHICH chunks
# come back, and in what order - not the numbers themselves.

## An honest result

**You may find embeddings do not clearly win here.** That is not a broken notebook — it is a real measurement.

Six different phrasings were tested across two embedding models. Neither reliably beat the simpler method on this corpus, and sometimes they did worse. The right clause *is* in there; it just ranks below other attendance clauses that look equally plausible to both.

This is the honest state of the field. A newer, fancier technique is not automatically better on **your** data, and you cannot tell from one example.

That is exactly why step 11 exists.

---

## Ask, using the new retriever

In [ ]:
print(grounded(by_meaning)(question, k=4))

---

## Now change it yourself

Hunt for a question where the two genuinely disagree. Change the question cell above and re-run the comparison.

Worth trying:

- `"What if I am sick and cannot attend for two weeks?"`
- `"Can I be let off for poor attendance?"`
- `"I was representing the college at a sports event"`

Keep a note of any where meaning-matching clearly wins. You will need evidence like that to justify the extra complexity to anyone.

---

# Step 8 — A workflow

You will build a fixed pipeline — a flowchart — that reads a student email, extracts fields from it, and routes it.

**What you should end up understanding:** How a model becomes part of ordinary software: it turns prose into JSON that an `if` statement can act on.

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#f9a825">&#9888; Uses about 4 API calls. Re-running cells is fine, it just uses a little more quota.</b></div>

So far the model answers questions. Real systems **do things** — classify, extract, route, decide.

A workflow is a flowchart. **You have been drawing these since first year.** The only new part is that a model sits inside one of the boxes.

In [ ]:
import json, re
from labcore import chat, corpus, embed, grounded

chunks, texts, _ = corpus()
ask = grounded(embed(texts))

print("ready")

### The incoming message

In [ ]:
# ===== EDIT ME, then run the cell below =====
email = ("Hi, I'm 24BCE0142. I had 68% attendance in Operating Systems "
         "because of placement drives. Will I be allowed to write the end "
         "sem? Please help urgently.")

### Box 1 — the model's only job: turn prose into JSON

An `if` statement cannot read a paragraph. It needs fields.

In [ ]:
def box1_extract(message):
    raw = chat([{"role": "user", "content":
        "Extract fields from this student message. Reply with ONLY a JSON "
        "object and no prose:\n"
        '{"topic": "attendance|grading|degree|other", '
        '"roll_no": "<id or null>", "course": "<name or null>", '
        '"urgent": true|false}\n\n' + message}])
    return json.loads(re.search(r"\{.*\}", raw, re.S).group())


ticket = box1_extract(email)
print(ticket)

**That is structured output.** It is most of the real work in a production pipeline, and it is taught almost nowhere.

### Boxes 2, 3 and 4 — plain Python

No model here at all. Your code decides.

In [ ]:
def handle(message):
    ticket = box1_extract(message)                    # box 1: the model

    if ticket["topic"] == "other":                    # box 2: your code
        return ticket, "ROUTED TO HUMAN - not a regulations question"

    answer = ask(message, k=6)                        # box 3: RAG

    if ticket["urgent"]:                              # box 4: your code
        answer = "[FLAGGED URGENT]\n" + answer

    return ticket, answer


ticket, outcome = handle(email)
print(ticket)
print()
print(outcome[:600])

### Now one that should never reach the answering model

In [ ]:
# ===== EDIT ME, then run the cell below =====
email = "can someone tell me the mess timings for saturday"

In [ ]:
ticket, outcome = handle(email)
print(ticket)
print(outcome)

Classified as `other` and routed to a human — **without ever reaching the RAG step.**

## Why this pattern dominates in industry

Same path every time, so you can test it, cost it, and debug it. **When it breaks, you know which box.**

The price: it only does what you drew. Anything outside the diagram falls straight through.

---

## Now change it yourself

Write your own message in the cell below and watch how it gets classified and routed.

In [ ]:
# ===== EDIT ME, then run the cell below =====
email = "I got 18 out of 60 in my DBMS internals. Am I finished?"

# Things worth trying:
#   - a message with no roll number at all. What does box 2 do?
#   - a very angry message. Does 'urgent' flip to true?
#   - a message about two things at once. Which topic wins?
#   - something in Telugu or Hindi. Does box 1 still return valid JSON?

In [ ]:
ticket, outcome = handle(email)
print(ticket)
print()
print(outcome[:700])

**That last one is worth doing.** Box 1 is the fragile part of every workflow like this — the moment it returns something that is not valid JSON, the whole pipeline throws. Try to break it.

---

# Step 11 — Measure it

You will score the system against ten questions whose correct answers you already know, then change one setting and score it again.

**What you should end up understanding:** How to tell whether a change made things better, with a number instead of an opinion. Almost nobody does this.

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; This step makes about 30 API calls and takes several minutes. On a free key that is a large chunk of your quota.</b></div>

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; Run the scoring cell once, read the results properly, and only re-run it after you have deliberately changed something. The comparison between two runs is the whole point — a single score on its own tells you nothing.</b></div>

Everyone demos on the three questions that work. **The ones who ship, measure.**

This is the step that gets you hired, and almost nobody does it.

In [ ]:
from labcore import corpus, tfidf, embed, grounded

chunks, texts, _ = corpus()
print("ready")

### The test set

Ten questions whose correct answers we already know, checked by hand against the PDF. Five categories, two each.

In [ ]:
# ===== EDIT ME, then run the cell below =====
EVAL = [
    ("What is the minimum attendance required in each course?", "75", "lookup"),
    ("How many credits are required for the B.Tech degree?", "160", "lookup"),
    ("I have 68% attendance because of placement drives. What happens?",
     ["condon", "10"], "two-docs"),
    ("I scored 18 out of 60 in formative assessment. What grade do I get?",
     ["R", "21"], "two-docs"),
    ("What are the hostel mess timings?", "REFUSE", "not-in-corpus"),
    ("How much is the tuition fee per semester?", "REFUSE", "not-in-corpus"),
    ("Can I get an exemption if I miss too many classes?", "condon", "paraphrased"),
    ("What do I need to score to not fail the internals?", "21", "paraphrased"),
    ("My attendance is exactly 75%. Am I eligible for the end sem?", "75", "edge"),
    ("My CGPA is exactly 7.0. What class do I get?", "distinction", "edge"),
]

print(f"{len(EVAL)} test questions")

Look at rows 5 and 6: **the correct answer is a refusal.** Refusing correctly scores a pass.

And rows 9 and 10 sit exactly on a boundary — *exactly* 75%, *exactly* 7.0. Those are the questions a real student actually worries about, and they are where wording matters more than numbers.

### The scorer

In [ ]:
def grade(answer, expected):
    low = answer.lower()
    if expected == "REFUSE":
        return "could not find" in low
    needed = expected if isinstance(expected, list) else [expected]
    return all(n.lower() in low for n in needed)


def score(ask, k, label):
    passed = 0
    print(f"\n{label}")
    for question, expected, category in EVAL:
        ok = grade(ask(question, k=k), expected)
        passed += ok
        print(f"  {'PASS' if ok else 'FAIL'}  [{category:13}] {question[:48]}")
    print(f"  --> {passed}/{len(EVAL)}")
    return passed


print("scorer defined - no API calls yet")

### Score one configuration

Each run of the cell below makes **10 API calls** and takes about half a minute.

In [ ]:
# ===== EDIT ME, then run the cell below =====
retriever = "tfidf"      # "tfidf" or "embed"
k = 1                    # how many clauses to retrieve

In [ ]:
r = tfidf(texts) if retriever == "tfidf" else embed(texts)
result = score(grounded(r), k, f"{retriever}, k={k}")

### Now change ONE thing and score again

Go back to the settings cell, change `k` to `6`, and run the scoring cell again.

**Write both numbers down.** The comparison is the entire point — a single score on its own tells you nothing.

---

## Read the failures, not just the score

Look at **which categories** failed.

If they cluster in `paraphrased`, your problem is retrieval and a bigger model will not help. If they cluster in `two-docs`, raise `k`. If `not-in-corpus` fails, your system is inventing and the system prompt needs work.

**That is a diagnosis with evidence behind it**, which is a completely different thing from a hunch.

---

## The sentence that gets you hired

> *"I built a RAG chatbot."* — everyone says this.
>
> *"I measured it at 60% on my own eval set, found the failures were retrieval not generation, and got it to 85%."* — almost nobody says this.

You have just done the second one in miniature. **Keep the numbers.**

---

## Now change it yourself

Add a question you can verify in your own regulations.

In [ ]:
# ===== EDIT ME, then run the cell below =====
EVAL = EVAL + [
    # (question, text that must appear in a correct answer, category)
    ("How long do I have to complete the degree?", "seven", "lookup"),
]

print(f"{len(EVAL)} questions now")

Then re-run the scoring cell above.

**Ten questions of your own, on documents you care about, is a weekend project and an interview answer.**

---

# What you built, and what is missing

You went from a bare API call to a system that answers questions about your own regulations, refuses when it should, and can be **measured**.

**What is missing**, honestly:

- **A real interface.** This is a notebook, not a product.
- **Fresh documents.** The regulations change; nothing here notices.
- **Access control.** Anyone can ask anything. In a real system, who is asking decides what they may see.
- **Agents** — letting the model choose its own steps instead of you fixing them. That is the next thing to learn.

---

## This weekend

Point this at something **you** care about — your department's rules, previous years' question papers, a lab manual.

Two rules decide whether the project teaches you anything:

1. **You must be able to check the answers yourself.** The test questions are the point, not the chatbot.
2. **The model must not already know the content.** Ask your question with no context first, the way step 2 does. If it answers correctly from memory, that document is useless as a test.

Then write ten questions with known answers, score it, change one thing, score it again.

> *"I built a RAG chatbot"* is what everyone says.
>
> *"I measured it at 60%, found the failures were retrieval not generation, and got it to 85%"* is what almost nobody says.

**Keep the numbers.**